In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
onshore_shapefile = snakemake.input.onshore_shapefile
geodata_file = snakemake.input.geodata_file
year = int(snakemake.wildcards.year)

plants_in_europe_path = snakemake.output.plants_in_europe
pcap_in_eur_year_path = snakemake.output.pcap_in_eur_year


In [ ]:
# get shapefiles from intermediate data from highres WF

# Load onshore shapefile
euroshape = gpd.read_file(onshore_shapefile).set_index(["index"])

In [ ]:
# open geopandas file with geometries
techs_plot = (
    gpd.read_file(geodata_file)
    .to_crs("epsg:4326")
)


In [ ]:
## plot of each plant

fig, ax = plt.subplots(1, 2, figsize=(15, 10))

euroshape.plot(ax=ax[0], facecolor="none", edgecolor="black", linewidth=0.5)
techs_plot.query("status=='operating'").query("Technology == 'solar'").plot(
    ax=ax[0], markersize=1, color="gold", label="Solar"
)
techs_plot.query("status=='operating'").query("Technology == 'onshore'").plot(
    ax=ax[0], markersize=1, color="tab:green", label="Onshore"
)
techs_plot.query("Technology == 'offshore'").plot(
    ax=ax[0], markersize=1, color="tab:blue", label="Offshore"
)
techs_plot.query("status=='operating'").query("Technology == 'nuclear'").plot(
    ax=ax[0], markersize=1, color="tab:red", label="Nuclear"
)
ax[0].set_title("Installed power plants 2025 (Nuclear 2024)")
ax[0].legend()

euroshape.plot(ax=ax[1], facecolor="none", edgecolor="black", linewidth=0.5)
techs_plot.query("Technology == 'solar' and estimated_retirement_year>=@year").plot(
    ax=ax[1], markersize=1, color="gold", label="Solar"
)
techs_plot.query("Technology == 'onshore' and estimated_retirement_year>=@year").plot(
    ax=ax[1], markersize=1, color="tab:green", label="Onshore"
)
techs_plot.query("Technology == 'offshore' and estimated_retirement_year>=@year").plot(
    ax=ax[1], markersize=1, color="tab:blue", label="Offshore"
)
techs_plot.query("Technology == 'nuclear' and estimated_retirement_year>=@year").plot(
    ax=ax[1], markersize=1, color="tab:red", label="Nuclear"
)
ax[1].set_title(f"Installed power plants {year}")
ax[1].legend()

fig.savefig(plants_in_europe_path, dpi=300, format="png", bbox_inches="tight")


In [ ]:
## For plotting
installed_cap = (
    techs_plot.assign(capacity_GW=techs_plot["capacity_MW"].div(1000))
    .query("status=='operating'")
    .loc[:, ["Technology", "index", "capacity_GW"]]
    .set_index("index")
)

installed_cap = installed_cap.merge(
    euroshape.loc[:, ["geometry"]],
    left_on="index",
    right_on="index",
    how="left",
)

installed_cap = gpd.GeoDataFrame(installed_cap)

onshore_cap = (
    installed_cap.query("Technology == 'onshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

offshore_cap = (
    installed_cap.query("Technology == 'offshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

solar_cap = (
    installed_cap.query("Technology == 'solar'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

nuclear_cap = (
    installed_cap.query("Technology == 'nuclear'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

In [ ]:
## For plotting
installed_cap_year = (
    techs_plot.assign(capacity_GW=techs_plot["capacity_MW"].div(1000))
    .query("estimated_retirement_year>=@year")
    .loc[:, ["Technology", "index", "capacity_GW"]]
    .set_index("index")
)


installed_cap_year = installed_cap_year.merge(
    euroshape.loc[:, ["geometry"]],
    left_on="index",
    right_on="index",
    how="left",
)

installed_cap_year = gpd.GeoDataFrame(installed_cap_year)

onshore_cap_year = (
    installed_cap_year.query("Technology == 'onshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

offshore_cap_year = (
    installed_cap_year.query("Technology == 'offshore'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

solar_cap_year = (
    installed_cap_year.query("Technology == 'solar'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

nuclear_cap_year = (
    installed_cap_year.query("Technology == 'nuclear'")
    .drop(columns=["Technology"])
    .dissolve(by="index", aggfunc="sum")
)

In [ ]:
## plot capacities by region


vmax_on = max(max(onshore_cap.capacity_GW), max(onshore_cap_year.capacity_GW))
vmax_off = max(max(offshore_cap.capacity_GW), max(offshore_cap_year.capacity_GW))
vmax_solar = max(max(solar_cap.capacity_GW), max(solar_cap_year.capacity_GW))
vmax_nuclear = max(max(nuclear_cap.capacity_GW), max(nuclear_cap_year.capacity_GW))


fig, ax = plt.subplots(2, 4, figsize=(20, 12))

## Existing capacities

# onshore
cmap = plt.get_cmap("magma_r")
onshore_cap.plot(
    column="capacity_GW",
    ax=ax[0][0],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_on,
)
euroshape.plot(ax=ax[0][0], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][0].set_title("Onshore capacities (GW) - 2025", fontsize=12, weight="bold")

# offshoregeodata_files["onshore"]
cmap = plt.get_cmap("magma_r")
offshore_cap.plot(
    column="capacity_GW",
    ax=ax[0][1],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_off,
)
euroshape.plot(ax=ax[0][1], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][1].set_title("Offshore capacities (GW) - 2025", fontsize=12, weight="bold")

# solar
cmap = plt.get_cmap("magma_r")
solar_cap.plot(
    column="capacity_GW",
    ax=ax[0][2],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_solar,
)
euroshape.plot(ax=ax[0][2], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][2].set_title("Solar capacities (GW) - 2025", fontsize=12, weight="bold")

# nuclear
cmap = plt.get_cmap("magma_r")
nuclear_cap.plot(
    column="capacity_GW",
    ax=ax[0][3],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_nuclear,
)
euroshape.plot(ax=ax[0][3], facecolor="none", edgecolor="black", linewidth=0.5)
ax[0][3].set_title("Nuclear capacities (GW) - 2024", fontsize=12, weight="bold")

## Estimated existing capacities

# onshore
cmap = plt.get_cmap("magma_r")
onshore_cap_year.plot(
    column="capacity_GW",
    ax=ax[1][0],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_on,
)
euroshape.plot(ax=ax[1][0], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][0].set_title(f"Onshore capacities (GW) - {year}", fontsize=12, weight="bold")

# offshore
cmap = plt.get_cmap("magma_r")
offshore_cap_year.plot(
    column="capacity_GW",
    ax=ax[1][1],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_off,
)
euroshape.plot(ax=ax[1][1], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][1].set_title(f"Offshore capacities (GW) - {year}", fontsize=12, weight="bold")

# solar
cmap = plt.get_cmap("magma_r")
solar_cap_year.plot(
    column="capacity_GW",
    ax=ax[1][2],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_solar,
)
euroshape.plot(ax=ax[1][2], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][2].set_title(f"Solar capacities (GW) - {year}", fontsize=12, weight="bold")

# nuclear
cmap = plt.get_cmap("magma_r")
nuclear_cap_year.plot(
    column="capacity_GW",
    ax=ax[1][3],
    markersize=0.5,
    cmap=cmap,
    legend=True,
    vmin=0,
    vmax=vmax_nuclear,
)
euroshape.plot(ax=ax[1][3], facecolor="none", edgecolor="black", linewidth=0.5)
ax[1][3].set_title(f"Nuclear capacities (GW) - {year}", fontsize=12, weight="bold")


plt.tight_layout()

fig.savefig(pcap_in_eur_year_path, dpi=300, format="png", bbox_inches="tight")